# NanoCore-S1 baseline — typed decisions on frozen EmbeddingGemma 2

The Experimentation pass for `docs/DECISION-MODEL.md`. Every candidate is scored
against honest baselines on a shared holdout:

- **D1, text Choice** — Banking77 (77 intents). Baselines: majority class,
  zero-shot cosine (free with the encoder), kNN. Candidates: `fingerprint` and
  `interaction` heads trained on the log score.
- **D2, image Choice** — CIFAR-10 subset through the *same* encoder + head
  pipeline: the multimodal claim, minimally.
- **Honesty controls** — shuffled-label and zeroed-head runs that *must* score
  at chance; if they do not, the evaluation is broken, not the model
  (the audit's synthetic-metric lesson).
- **Composer** — a plumbing check on real EG2 vectors. Explicitly *not* a
  capability claim: the composer needs multi-item states, which neither dataset
  provides (D3).

## 0. Environment

In [ ]:
import subprocess, sys, time, json
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout
print(gpu)
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert "T4" in gpu, f"Expected T4, got {gpu!r} — set machine_shape to NvidiaTeslaT4"

STATUS = {"status": "running", "stage": "env", "started": time.time()}
def write_status(stage=None, error=None, done=False):
    if stage: STATUS["stage"] = stage
    if error: STATUS["error"] = str(error); STATUS["status"] = "failed"
    if done: STATUS["status"] = "complete"
    STATUS["ended"] = time.time()
    Path("/kaggle/working/run_status.json").write_text(json.dumps(STATUS, indent=2))
write_status()

In [ ]:
# Self-contained source bundle. The worktree input dataset previously failed at
# Kaggle's mount service before Python started (ERRORED_MOUNTING_DATASET), so the
# decision package and its contract tests are embedded in this notebook revision.
import base64, io, sys, zipfile
from pathlib import Path

BUNDLE = "..."
REPO = "/kaggle/working/nanocore-s1"
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as z:
    z.extractall(REPO)
sys.path.insert(0, REPO)
print("self-contained source extracted:", REPO)

In [ ]:
# Dependency floors that matter (jev-stack AGENTS.md):
# sentence-transformers>=6.1.0 for EG2; transformers>=5.19.0 for embedding_gemma2.
# NO -U flag and NO torchvision: force-upgrading already-satisfied deps on this
# image pulls binaries mismatched to its custom torch 2.11 build and breaks the
# torch install outright (v1 autopsy: ImportError in torch.distributed).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sentence-transformers>=6.1.0", "transformers>=5.19.0", "datasets"],
               check=True)
import torch, transformers, sentence_transformers
print("transformers", transformers.__version__,
      "| sentence-transformers", sentence_transformers.__version__,
      "| torch still healthy:", torch.cuda.is_available())

In [ ]:
# Contract tests inside the kernel — cheap proof the tree works on this image.
import subprocess, sys
from pathlib import Path
r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_decision_schema.py",
                    "tests/test_decision_head.py", "tests/test_decision_composer.py",
                    "tests/test_decision_model.py", "-x", "-q"],
                   capture_output=True, text=True, cwd=REPO)
print(r.stdout[-2000:]); print(r.stderr[-1000:])
assert r.returncode == 0, "contract tests failed on T4"
write_status("contract-tests")

## 1. Encoder — one load for both demos

`modalities=("text", "vision")` loads text+vision at 440M params; it encodes text
too, so one model serves Banking77 *and* CIFAR-10. bf16 on the T4 — never fp16.

In [ ]:
import torch, numpy as np
from src.decision.encoder import StateEncoder, to_numpy

encoder = StateEncoder(modalities=("text", "vision"), device="cuda")
n_params = sum(p.numel() for p in encoder.model.parameters())
print(f"parameters: {n_params/1e6:.0f}M | dtype: {encoder.dtype} | dim: {encoder.embedding_dim}")
write_status("encoder-loaded")

## 2. D1 — Banking77 text Choice

States use the symmetric `Classification` prompt; the 77 intent names (humanized)
are embedded once as `Document`. Train split → 90% fit / 10% calibration.

In [ ]:
from datasets import load_dataset

def load_hf(repo):
    # datasets>=4 rejects legacy loading scripts; HF's auto-parquet conversion
    # under refs/convert/parquet is the supported path for those repos.
    try:
        return load_dataset(repo)
    except Exception as e:
        print(f"main ref failed ({type(e).__name__}), trying refs/convert/parquet")
        return load_dataset(repo, revision="refs/convert/parquet")

# banking77 upstream is script-only and has no parquet branch. Verified from
# the script itself: it pulls two raw CSVs from GitHub (text + label NAME), and
# the canonical 77-label order is its names=[...] list — parse both from source.
import ast, requests, pandas as pd

RAW = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/"
script = requests.get(
    "https://huggingface.co/datasets/PolyAI/banking77/resolve/main/banking77.py").text
i, j = script.index("names=["), script.index("]", script.index("names=["))
label_names = ast.literal_eval(script[i + 6:j + 1])
train_df = pd.read_csv(RAW + "train.csv")
test_df  = pd.read_csv(RAW + "test.csv")
print("csv columns:", train_df.columns.tolist(), "| n_labels:", len(label_names))

text_col, label_col = train_df.columns[0], train_df.columns[1]
def to_idx(v):
    return label_names.index(v) if isinstance(v, str) else int(v)
option_texts = [n.replace("_", " ") for n in label_names]
print(f"train {len(train_df)} | test {len(test_df)} | {len(label_names)} intents")
print("examples:", option_texts[:5])

STATE_PROMPT, OPTION_PROMPT = "Classification", "Document"
train_texts = train_df[text_col].tolist()
train_y = np.array([to_idx(v) for v in train_df[label_col]])
test_texts = test_df[text_col].tolist()
test_y = np.array([to_idx(v) for v in test_df[label_col]])

g = np.random.default_rng(0)
perm = g.permutation(len(train_texts))
n_val = int(0.1 * len(perm))
val_idx, fit_idx = perm[:n_val], perm[n_val:]

print("encoding states+options…", flush=True)
X_train = to_numpy(encoder.encode(train_texts, prompt_name=STATE_PROMPT, batch_size=64))
X_test  = to_numpy(encoder.encode(test_texts,  prompt_name=STATE_PROMPT, batch_size=64))
O       = to_numpy(encoder.encode_options(option_texts, prompt_name=OPTION_PROMPT))
print("shapes:", X_train.shape, X_test.shape, O.shape)

X_fit, y_fit = X_train[fit_idx], train_y[fit_idx]
X_val, y_val = X_train[val_idx], train_y[val_idx]
np.savez_compressed("/kaggle/working/banking77_embeddings.npz",
                    X_train=X_train, y_train=train_y, X_test=X_test, y_test=test_y,
                    O=O, option_texts=np.array(option_texts))
write_status("d1-encoded")

### Baselines — every candidate must beat these

The **zero-shot cosine** row is the real baseline: `argmax_j cos(s, o_j)` costs
nothing beyond encoding, and it is what an untrained deployment would use.

In [ ]:
from src.decision import metrics

RESULTS = {"d1": {}, "d2": {}, "controls": {}}

def acc(pred, y): return float(np.mean(np.asarray(pred) == np.asarray(y)))

# majority class
maj = np.bincount(y_fit).argmax()
RESULTS["d1"]["majority"] = {"accuracy": acc(np.full(len(test_y), maj), test_y)}

# zero-shot cosine: argmax over option embeddings
zs = (X_test @ O.T).argmax(1)
RESULTS["d1"]["zero_shot_cosine"] = {"accuracy": acc(zs, test_y)}

# kNN over train state embeddings (unit vectors → argmax cosine)
for k in (1, 5):
    sims = X_test @ X_fit.T
    topk = sims.argsort(1)[:, -k:]
    pred = np.array([np.bincount(y_fit[row], minlength=len(O)).argmax() for row in topk])
    RESULTS["d1"][f"knn_{k}"] = {"accuracy": acc(pred, test_y)}

for name, r in RESULTS["d1"].items():
    print(f"{name:22s} acc={r['accuracy']:.4f}")

### Trained heads — fingerprint and interaction

Fit on the log score (strictly proper), temperature on the held-out 10%,
abstention threshold fitted for 90% selective precision. Reported on test:
accuracy, log score, Brier, ECE.

In [ ]:
from src.decision.head import DecisionHead
from src.decision.schema import DecisionExample

labels = list(option_texts)
def make_examples(X, y):
    return [DecisionExample(state_embedding=X[i], qtype="choice", labels=labels,
                            target=[1.0 if j == y[i] else 0.0 for j in range(len(labels))],
                            option_embeddings=[O[j] for j in range(len(labels))])
            for i in range(len(X))]

fit_ex = make_examples(X_fit, y_fit)
val_ex = make_examples(X_val, y_val)

def evaluate_head(head, tag, epochs=120):
    head.fit(fit_ex, epochs=epochs, lr=0.05, batch_size=64, seed=0)
    head.fit_temperature(val_ex)
    tau = head.fit_abstention_threshold(val_ex, "choice", target_precision=0.9)
    probs, preds, kept_scores = [], [], []
    for i in range(len(X_test)):
        p = head.predict(X_test[i], [O[j] for j in range(len(labels))],
                         qtype="choice", labels=labels)
        probs.append([p.probabilities[l] for l in labels])
        preds.append(p.choice); kept_scores.append(p.max_score)
    y_onehot = [[1.0 if j == test_y[i] else 0.0 for j in range(len(labels))]
                for i in range(len(test_y))]
    r = {"accuracy": acc([labels.index(c) for c in preds], test_y),
         "log_score": metrics.weighted_metric(probs, y_onehot, metrics.log_score),
         "brier": metrics.weighted_metric(probs, y_onehot, metrics.brier_score),
         "ece": metrics.weighted_metric(probs, y_onehot, metrics.expected_calibration_error),
         "temperature": head.temperature.get("choice"),
         "abstention_tau": tau}
    RESULTS["d1"][tag] = r
    print(f"{tag:22s} acc={r['accuracy']:.4f} log={r['log_score']:.3f} "
          f"brier={r['brier']:.3f} ece={r['ece']:.3f} T={r['temperature']:.2f}")
    return head

head_fp  = evaluate_head(DecisionHead(dim=768, mode="fingerprint"), "fingerprint", epochs=150)
head_int = evaluate_head(DecisionHead(dim=768, mode="interaction", hidden=64), "interaction", epochs=60)
head_fp.save("/kaggle/working/head_fingerprint.json")
head_int.save("/kaggle/working/head_interaction.json")
write_status("d1-heads")

### Honesty controls — the audit's lesson

Two runs that *must* score at chance (1/77 ≈ 0.013). If either does not, the
pipeline is leaking and every number above is void.

In [ ]:
# 1. shuffled targets — no real signal should remain
g2 = np.random.default_rng(1)
shuf = make_examples(X_fit, y_fit[g2.permutation(len(y_fit))])
h = DecisionHead(dim=768, mode="fingerprint")
h.fit(shuf, epochs=120, lr=0.05, batch_size=64, seed=0)
shuf_pred = [h.predict(X_test[i], None, qtype="choice", labels=labels).choice
             for i in range(len(X_test))]
shuf_acc = acc([labels.index(c) for c in shuf_pred], test_y)
RESULTS["controls"]["shuffled_labels"] = {"accuracy": shuf_acc, "expect": "~0.013"}

# 2. zeroed head — a dead model must not score well (the audit's ECE trap)
hz = DecisionHead(dim=768, mode="fingerprint")
hz.fit(fit_ex[:64], epochs=1)  # register fingerprints
with torch.no_grad():
    for v in hz.parameters().values():
        v.zero_()
zero_pred = [hz.predict(X_test[i], None, qtype="choice", labels=labels).choice
             for i in range(len(X_test))]
zero_acc = acc([labels.index(c) for c in zero_pred], test_y)
RESULTS["controls"]["zeroed_head"] = {"accuracy": zero_acc, "expect": "~0.013"}

print(f"shuffled: {shuf_acc:.4f} | zeroed: {zero_acc:.4f}  (both should be ≈ 1/77)")
ok = shuf_acc < 0.05 and zero_acc < 0.05
RESULTS["controls"]["evaluation_sane"] = ok
print("evaluation sane:", ok)
Path("/kaggle/working/d1_results.json").write_text(json.dumps(RESULTS, indent=2, default=str))
write_status("controls")

## 3. D2 — CIFAR-10 image Choice (the multimodal claim, minimally)

Same pipeline, different state item: `{"image": PIL}` through the vision
encoder. Zero-shot cosine against the 10 class names, then a fingerprint head
on 10 shots/class. Small subsets — this is a smoke test, not a benchmark claim.

In [ ]:
from datasets import load_dataset

cifar = load_hf("uoft-cs/cifar10")
classes = cifar["train"].features["label"].names
option_texts_img = [f"a photo of a {c}" for c in classes]

def subset(split, per_class):
    seen, idx = {}, []
    for i, y in enumerate(split["label"]):
        if seen.get(y, 0) < per_class:
            idx.append(i); seen[y] = seen.get(y, 0) + 1
    return idx

fit_i, test_i = subset(cifar["train"], 10), subset(cifar["test"], 10)
print(f"fit {len(fit_i)} | test {len(test_i)} — encoding images in microbatches…", flush=True)
torch.cuda.empty_cache()

try:
    Xi_fit = to_numpy(encoder.encode([{"image": cifar["train"][i]["img"]} for i in fit_i], batch_size=2))
    Xi_test = to_numpy(encoder.encode([{"image": cifar["test"][i]["img"]} for i in test_i], batch_size=2))
except torch.cuda.OutOfMemoryError as e:
    torch.cuda.empty_cache()
    write_status("d2-image-oom", error=e)
    raise
print("vision peak allocated GiB:", round(torch.cuda.max_memory_allocated() / 2**30, 2))
Oi = to_numpy(encoder.encode_options(option_texts_img, prompt_name="Document"))
yi_fit = np.array([cifar["train"][i]["label"] for i in fit_i])
yi_test = np.array([cifar["test"][i]["label"] for i in test_i])

zs_img = (Xi_test @ Oi.T).argmax(1)
RESULTS["d2"]["zero_shot_cosine"] = {"accuracy": acc(zs_img, yi_test)}

labels_img = list(option_texts_img)
img_ex = [DecisionExample(state_embedding=Xi_fit[i], qtype="choice", labels=labels_img,
                          target=[1.0 if j == yi_fit[i] else 0.0 for j in range(10)])
          for i in range(len(Xi_fit))]
hi = DecisionHead(dim=768, mode="fingerprint")
hi.fit(img_ex, epochs=120, lr=0.05, batch_size=32, seed=0)
img_pred = [hi.predict(Xi_test[i], None, qtype="choice", labels=labels_img).choice
            for i in range(len(Xi_test))]
RESULTS["d2"]["fingerprint_10shot"] = {"accuracy": acc([labels_img.index(c) for c in img_pred], yi_test)}

print(f"zero-shot cosine : {RESULTS['d2']['zero_shot_cosine']['accuracy']:.4f}")
print(f"fingerprint 10/class: {RESULTS['d2']['fingerprint_10shot']['accuracy']:.4f}")
write_status("d2")

## 4. Composer — plumbing check only

Real EG2 vectors through `EmbeddingComposer`: correct shapes, modality ids,
masking. **Not a capability claim** — multi-item states need D3 data.

In [ ]:
from src.decision.composer import ComposerConfig, EmbeddingComposer, MODALITY_IDS

composer = EmbeddingComposer(ComposerConfig(n_layer=4, n_head=6, n_embd=768))
print(f"composer parameters: {composer.n_parameters()/1e6:.1f}M")

# a two-item state: one text + one 'image' slot reusing a CIFAR embedding
two_items = torch.tensor(np.stack([X_test[0], Xi_test[0]])).unsqueeze(0).float()
mod_ids = torch.tensor([[MODALITY_IDS["text"], MODALITY_IDS["image"]]])
s = composer(two_items, mod_ids)   # composer runs on CPU — it is 28M params
assert s.shape == (1, 768) and torch.isfinite(s).all()
print("composed state:", tuple(s.shape), "finite:", bool(torch.isfinite(s).all()))
RESULTS["composer_smoke"] = {"shape": list(s.shape), "params": composer.n_parameters()}

In [ ]:
# Package artifacts — single results file + report; kernels download /kaggle/working.
import zipfile, json
report = ["# NanoCore-S1 baseline run", ""]
for section in ("d1", "d2", "controls", "composer_smoke"):
    report.append(f"## {section}")
    for k, v in RESULTS[section].items():
        report.append(f"- {k}: {v}")
    report.append("")
Path("/kaggle/working/REPORT.md").write_text("\n".join(report))
Path("/kaggle/working/results.json").write_text(json.dumps(RESULTS, indent=2, default=str))
write_status("done", done=True)
with zipfile.ZipFile("/kaggle/working/artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["results.json", "REPORT.md", "head_fingerprint.json",
              "head_interaction.json", "banking77_embeddings.npz", "run_status.json"]:
        p = Path("/kaggle/working") / f
        if p.exists(): z.write(p, f)
print(json.dumps(RESULTS, indent=2, default=str)[:3000])